# Séance 3 — Classification & Régression Logistique

**Formation ML — formation-ml**

### Objectifs

- Comprendre pourquoi la régression linéaire ne convient pas à un problème de classification
- Construire la régression logistique : sigmoïde + log-loss + descente de gradient
- Implémenter le tout **à la main** en NumPy, puis comparer à scikit-learn
- Visualiser une frontière de décision

---
<div style="background:#eef6ff;border-left:5px solid #1a73e8;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>💡 Intuition</b><br>
Même structure qu'en Séance 2 : ① un modèle, ② une fonction de coût, ③ une optimisation. Seul le contenu de chaque brique change.
</div>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

sns.set_theme(style="whitegrid")
np.random.seed(42)

## 1. Charger et explorer les données Titanic

In [ ]:
df = pd.read_csv("titanic.csv")
df.head()

In [ ]:
df.info()

<div style="background:#fff3e0;border-left:5px solid #e67e22;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>⚠️ Attention</b><br>
Remarque les valeurs manquantes sur <code>Age</code> et <code>Cabin</code> — normal, on y reviendra en détail en Séance 4 (imputation). Pour cette séance, on se contente de retirer les lignes sans <code>Age</code>.
</div>

In [ ]:
df_clean = df.dropna(subset=["Age"]).copy()
print(f"Lignes avant : {len(df)}, après suppression des Age manquants : {len(df_clean)}")

df_clean["Survived"].value_counts(normalize=True)

## 2. Pourquoi pas une droite ?

In [ ]:
x = df_clean["Fare"].values
y = df_clean["Survived"].values
mask = x < 150

plt.figure(figsize=(8, 5))
plt.scatter(x[mask], y[mask], alpha=0.3)
plt.xlabel("Tarif payé (Fare)")
plt.ylabel("Survécu (0/1)")
plt.title("La target est binaire, pas continue")
plt.show()

<div style="background:#eef6ff;border-left:5px solid #1a73e8;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>💡 Intuition</b><br>
Essaie d'imaginer une droite $\hat{y}=wx+b$ qui "passe au mieux" entre ces points. Rien ne l'empêche de sortir de $[0,1]$ pour des valeurs de $x$ extrêmes — pourtant on veut une <b>probabilité</b>, forcément entre 0 et 1. D'où la sigmoïde.
</div>

## 3. La sigmoïde, implémentée à la main

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z_test = np.linspace(-10, 10, 200)
plt.figure(figsize=(7, 4))
plt.plot(z_test, sigmoid(z_test), linewidth=2.5)
plt.axhline(0.5, color="gray", linestyle="--", linewidth=1)
plt.xlabel("z = wx + b")
plt.ylabel("σ(z)")
plt.title("La fonction sigmoïde")
plt.show()

<div style="background:#f3e5f5;border-left:5px solid #8e24aa;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>✏️ Exercice</b><br>
Vérifie que <code>sigmoid(0)</code> vaut bien 0.5, et que <code>sigmoid(10)</code> est très proche de 1.
</div>

In [ ]:
# TODO : teste sigmoid(0) et sigmoid(10) ici



### D'où vient vraiment la sigmoïde ?

On modélise en fait la **cote** (odds) $= \dfrac{p}{1-p}$, dont le logarithme (le **logit**) peut prendre n'importe quelle valeur réelle — contrairement à $p$, coincée entre 0 et 1 :

$$\log\left(\frac{p}{1-p}\right) = wx+b \quad\Longrightarrow\quad p = \sigma(wx+b)$$

<div style="background:#eef6ff;border-left:5px solid #1a73e8;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>💡 Intuition</b><br>
La sigmoïde n'est donc pas choisie arbitrairement pour sa jolie forme en S — c'est la conséquence directe de modéliser <b>linéairement</b> le log de la cote.
</div>

In [ ]:
# Vérification : le logit doit être une droite parfaite en fonction de z
z_check = np.linspace(-6, 6, 100)
p_check = sigmoid(z_check)
logit_check = np.log(p_check / (1 - p_check))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(z_check, p_check)
axes[0].set_title("p = σ(z)")
axes[0].set_xlabel("z")
axes[1].plot(z_check, logit_check, color="C1")
axes[1].set_title("log(p/(1-p)) en fonction de z — une droite")
axes[1].set_xlabel("z")
plt.tight_layout()
plt.show()

## 4. Le log-loss, implémenté à la main

### D'où vient la formule du log-loss ?

Le modèle prédit $\hat{y} = P(y=1\mid x)$. La probabilité qu'il attribue à ce qui s'est **réellement produit** s'écrit :

$$P(y\mid x) = \hat{y}^{\,y}(1-\hat{y})^{\,1-y}$$

On cherche les paramètres qui **maximisent** cette probabilité sur tout le dataset (produit sur les observations, indépendantes) :

$$\mathcal{L}(w,b) = \prod_i \hat{y}_i^{\,y_i}(1-\hat{y}_i)^{\,1-y_i}$$

En passant au log (transforme le produit en somme, ne change pas où se situe le maximum) puis en prenant l'opposé (pour minimiser plutôt que maximiser) :

$$\text{Log-loss} = -\frac{1}{n}\sum_i \left[y_i\log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i)\right]$$

<div style="background:#e8f5e9;border-left:5px solid #2e7d32;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>✅ À retenir</b><br>
Ce n'est pas une formule arbitraire — c'est la conséquence directe du <b>maximum de vraisemblance</b> appliqué à une sortie probabiliste binaire.
</div>

In [ ]:
def log_loss(y_true, y_pred, eps=1e-9):
    y_pred = np.clip(y_pred, eps, 1 - eps)  # évite log(0)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

# Sanity check : une prédiction parfaite doit donner un coût proche de 0
print("Coût si prédiction parfaite :", log_loss(np.array([1, 0]), np.array([0.999, 0.001])))
print("Coût si prédiction inversée :", log_loss(np.array([1, 0]), np.array([0.001, 0.999])))

<div style="background:#e8f5e9;border-left:5px solid #2e7d32;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>✅ À retenir</b><br>
Le deuxième cas (confiant et faux) doit donner un coût <b>beaucoup</b> plus élevé que le premier — c'est exactement le comportement recherché.
</div>

## 5. La descente de gradient, à la main

In [ ]:
def gradient_log_loss(w, b, X, y):
    n = len(y)
    z = w * X.flatten() + b
    y_pred = sigmoid(z)
    error = y_pred - y  # attention à l'ordre, contrairement à la régression linéaire
    dw = (1 / n) * np.sum(X.flatten() * error)
    db = (1 / n) * np.sum(error)
    return dw, db

def descente_de_gradient_logistique(X, y, w_init, b_init, lr, n_iterations):
    w, b = w_init, b_init
    historique_loss = []
    for _ in range(n_iterations):
        dw, db = gradient_log_loss(w, b, X, y)
        w -= lr * dw
        b -= lr * db
        y_pred = sigmoid(w * X.flatten() + b)
        historique_loss.append(log_loss(y, y_pred))
    return w, b, historique_loss

In [ ]:
X_fare = df_clean[["Fare"]].values
y_target = df_clean["Survived"].values

scaler = StandardScaler()
X_fare_scaled = scaler.fit_transform(X_fare)

w_final, b_final, loss_history = descente_de_gradient_logistique(
    X_fare_scaled, y_target, w_init=0.0, b_init=0.0, lr=0.5, n_iterations=200
)

print(f"w trouvé : {w_final:.4f}")
print(f"b trouvé : {b_final:.4f}")

plt.figure(figsize=(7, 4))
plt.plot(loss_history)
plt.xlabel("Itération")
plt.ylabel("Log-loss")
plt.title("Convergence de la descente de gradient (régression logistique)")
plt.show()

### Interpréter le coefficient : l'odds ratio

<div style="background:#eef6ff;border-left:5px solid #1a73e8;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>💡 Intuition</b><br>
Puisque $\log(\text{cote}) = wx+b$, augmenter $x$ de 1 (écart-type, ici standardisé) <b>multiplie la cote de survie par $e^w$</b> — c'est l'<b>odds ratio</b>, la grandeur réellement utilisée pour interpréter un modèle logistique en pratique.
</div>

In [ ]:
odds_ratio = np.exp(w_final)
print(f"w = {w_final:.3f}")
print(f"odds ratio (e^w) = {odds_ratio:.3f}")
print(f"-> chaque +1 écart-type de tarif multiplie la cote de survie par ~{odds_ratio:.2f}")

<div style="background:#f3e5f5;border-left:5px solid #8e24aa;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>✏️ Exercice</b><br>
Essaie plusieurs learning rates (0.01, 0.5, 3.0). Retrouves-tu les mêmes comportements qu'en Séance 2 (trop lent / correct / diverge) ?
</div>

In [ ]:
# TODO : teste d'autres learning rates ici




## 6. Généraliser à plusieurs features + scikit-learn

In [ ]:
features = ["Age", "Fare", "Pclass", "SibSp", "Parch"]
X_multi = df_clean[features].values
y_multi = df_clean["Survived"].values

X_train, X_test, y_train, y_test = train_test_split(
    X_multi, y_multi, test_size=0.2, random_state=42, stratify=y_multi
)

scaler_multi = StandardScaler()
X_train_scaled = scaler_multi.fit_transform(X_train)
X_test_scaled = scaler_multi.transform(X_test)

model = LogisticRegression()
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
print("Accuracy sur le test :", accuracy_score(y_test, y_pred))

<div style="background:#fff3e0;border-left:5px solid #e67e22;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>⚠️ Attention</b><br>
<code>stratify=y_multi</code> assure que la proportion de survivants est la même dans train et test — important quand les classes ne sont pas parfaitement équilibrées (ici ~38% de survivants).
</div>

In [ ]:
print(classification_report(y_test, y_pred, target_names=["N'a pas survécu", "A survécu"]))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Prédit: non", "Prédit: oui"],
            yticklabels=["Réel: non", "Réel: oui"])
plt.title("Matrice de confusion")
plt.show()

## 7. Visualiser une frontière de décision (2 features)

In [ ]:
X_2d = df_clean[["Age", "Fare"]].values
y_2d = df_clean["Survived"].values
mask_fare = X_2d[:, 1] < 150
X_2d, y_2d = X_2d[mask_fare], y_2d[mask_fare]

scaler_2d = StandardScaler()
X_2d_scaled = scaler_2d.fit_transform(X_2d)

model_2d = LogisticRegression()
model_2d.fit(X_2d_scaled, y_2d)

xx, yy = np.meshgrid(
    np.linspace(X_2d_scaled[:, 0].min() - 0.5, X_2d_scaled[:, 0].max() + 0.5, 200),
    np.linspace(X_2d_scaled[:, 1].min() - 0.5, X_2d_scaled[:, 1].max() + 0.5, 200)
)
Z = model_2d.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, levels=20, cmap="RdYlGn", alpha=0.5)
plt.contour(xx, yy, Z, levels=[0.5], colors="black", linewidths=2, linestyles="--")
plt.scatter(X_2d_scaled[:, 0], X_2d_scaled[:, 1], c=y_2d, cmap="RdYlGn",
            edgecolor="black", linewidth=0.3, s=30)
plt.xlabel("Âge (standardisé)")
plt.ylabel("Tarif (standardisé)")
plt.title("Frontière de décision")
plt.colorbar(label="P(survie)")
plt.show()

<div style="background:#f3e5f5;border-left:5px solid #8e24aa;padding:12px 16px;border-radius:4px;margin:10px 0;">
<b>✏️ Exercice</b><br>
Remplace <code>Age</code> et <code>Fare</code> par <code>Pclass</code> et <code>Fare</code>. La frontière change-t-elle de forme ? Qu'est-ce que ça te dit sur l'importance relative des deux features ?
</div>

In [ ]:
# TODO : refais le graphe avec Pclass et Fare




## 8. Débrief

| Brique | Régression (S2) | Classification (S3) |
|---|---|---|
| ① Modèle | $\hat{y}=wx+b$ | $\hat{y}=\sigma(wx+b)$ |
| ② Coût | MSE | Log-loss |
| ③ Optimisation | Descente de gradient | Descente de gradient (identique) |

La structure ①②③ tient toujours. **Séance 4** : comment être sûr que ce modèle généralise bien, au-delà de l'accuracy brute.

## 9. Auto-évaluation rapide

**À rendre sur Classroom** — réponds par écrit à chacune des questions ci-dessous (quelques phrases suffisent), dans les cellules de réponse ci-après, puis soumets ce notebook complet.

1. Pourquoi la MSE n'est-elle pas adaptée à un problème de classification, même si elle "marche" mathématiquement ?
2. Que représente $\hat{y}$ dans la régression logistique — une classe ou une probabilité ?
3. Qu'est-ce que la frontière de décision, géométriquement ?

### ✏️ Tes réponses

**1.** 

**2.** 

**3.** 